# 02 · FunnyBirds + CBM — grounding & mechanism  *(seed-aware)*

**Claim (CBM):** a part concept reflects its part. **Backwash:** it instead reads
species / the rest of the bird. Two probes: **deletion grounding** (causal) and the
**species probe** (mechanism). All cells aggregate over every available seed
(`funnybirds-cbm-s*`); error bars are std across seeds.
*Refs: `fb_cbm_renderer_swap_v2.ipynb`, `fb_cbm_counterfactual.ipynb` §6.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
EPS = 1e-3
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
# ---- seed-aware grounding loaders ----
def load_grounding(prefix):
    """All seeds for a config prefix -> one df with a 'seed' column (None if absent)."""
    fs = sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-s*.parquet")))
    if not fs: return None
    out=[]
    for f in fs:
        d=pd.read_parquet(f); d["seed"]=int(re.search(r"-s(\d+)\.parquet$", f).group(1)); out.append(d)
    return pd.concat(out, ignore_index=True)
def per_part_seedagg(df, visible_only=True):
    """per (seed,part) retained_frac -> per-part mean/std/count across seeds."""
    d = df[df["changed_frac"]>EPS] if (visible_only and "changed_frac" in df.columns) else df
    g = d.groupby(["seed","part"]).agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    g["rf"] = g.pr/g.pi
    return g["rf"].groupby("part").agg(["mean","std","count"])


## What a CBM is, and what `z` and `c_preds` are
`image x → encoder p(z|x) → z → concept head q(c|z) → c_preds → label head → y`
- **`z`** — per-concept bottleneck latent the encoder reads from the image (26 slots).
- **`c_preds`** — concept probabilities (concept head on `z`), the human-readable layer.
- **`y`** — the class, from `z`/concepts.
Backwash lives in **encoder→`z`**: does `z_j` read *its part's pixels* or the species?

## 0 · Training sanity & overfitting (seed 1, representative)
Per-epoch held-out accuracy from `results/funnybirds-cbm/1/predictions/epoch_*.pth`.

In [ ]:
import torch
preds = REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/"1"/"predictions"
def _acc(pth):
    d = torch.load(pth, map_location="cpu", weights_only=False)
    yp,y=d["y_preds"],d["y"]
    ta=(yp.argmax(-1)==y).float().mean().item() if yp.ndim>1 else (yp==y).float().mean().item()
    ca=None
    if d.get("c_preds") is not None and d.get("c") is not None:
        cp=d["c_preds"]; cp=cp[...,0] if cp.ndim==3 else cp; ca=((cp>=0.5).float()==d["c"]).float().mean().item()
    return ta,ca
fs=sorted(glob.glob(str(preds/"epoch_*.pth")), key=lambda p:int(re.findall(r"epoch_(\d+)",p)[0]))
if not fs: print(f"[pending] no per-epoch predictions in {preds}")
else:
    R=pd.DataFrame([(int(re.findall(r"epoch_(\d+)",f)[0]),*_acc(f)) for f in fs],
                   columns=["epoch","task","concept"]).sort_values("epoch"); best=int(R.loc[R.task.idxmax(),"epoch"])
    display(R.round(4)); fig,ax=plt.subplots(figsize=(6,3.4))
    ax.plot(R.epoch,R.task,"o-",color=CBM_C,label="task (val)")
    if R.concept.notna().any(): ax.plot(R.epoch,R.concept,"s--",color="#5B8C5A",label="concept (val)")
    ax.axvline(best,ls=":",color="k"); ax.set_xlabel("epoch"); ax.set_ylabel("val acc")
    ax.set_title(f"Training curve — best task epoch {best}"); ax.legend()
    drop=R.task.max()-R.task.iloc[-1]
    print("VERDICT:", "plateau -> no overfit" if abs(drop)<0.01 else f"peaks {best} then declines")

## 1 · Deletion grounding — per-part `retained_frac` (all vs visible-only, ±seed std)
`retained_frac = P(removed)/P(intact)`. **visible-only** (headline) drops no-op removals
(part occluded in the intact image). `conf_on_occluded` = P(concept) the model asserts
for parts that are occluded in the intact image — a second, deletion-free backwash signal.

In [ ]:
G = load_grounding("funnybirds-cbm")
if G is None: print("[pending] bash analysis/grounding_sweep.sh")
else:
    allp = per_part_seedagg(G, visible_only=False)
    visp = per_part_seedagg(G, visible_only=True)
    R = pd.DataFrame({"retained_all":allp["mean"], "retained_visible":visp["mean"],
                      "vis_std":visp["std"]})
    if "changed_frac" in G.columns:
        occ = G[G["changed_frac"]<=EPS]
        R["conf_on_occluded"] = occ.groupby("part").p_intact.mean()
        R["frac_noop"] = G.groupby("part").changed_frac.apply(lambda s:(s<=EPS).mean())
    R = R.sort_values("retained_visible", ascending=False); display(R.round(3))
    print(f"n_seeds = {G.seed.nunique()}  (seeds: {sorted(G.seed.unique())})")
    fig,ax=plt.subplots(figsize=(6.6,3.4)); x=np.arange(len(R)); w=0.4
    ax.bar(x-w/2, R.retained_all, w, color="#9ec9e2", label="all removals (inflated)")
    ax.bar(x+w/2, R.retained_visible, w, yerr=R.vis_std.fillna(0), capsize=3, color=CBM_C, label="visible-only (headline)")
    ax.set_xticks(x); ax.set_xticklabels(R.index, rotation=30, ha="right")
    ax.set_ylabel("retained_frac"); ax.set_ylim(0,1); ax.legend()
    ax.set_title("FunnyBirds · CBM · removed-part concept retention")

## 2 · Species-identity probe — is the bottleneck a class code? (seed-averaged)
`species←c_preds` = how much the reported concepts alone pin the species (chance 1/50).
Per-part = species from each part's concept block alone. *(species←c_preds≈1 is partly
tautological with clean concepts; the per-part numbers are the informative ones.)*

In [ ]:
sps = sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-cbm-s*.json")))
PARTV=None
if not sps: print("[pending] bash analysis/grounding_sweep.sh")
else:
    Ss=[json.loads(Path(p).read_text()) for p in sps]; ch=Ss[0]["chance"]
    zc=np.mean([s["species_from_z"]["acc"] for s in Ss]); cc=np.mean([s["species_from_cpreds"]["acc"] for s in Ss])
    print(f"n_seeds={len(Ss)} | chance={ch:.3f} | species<-z {zc:.3f} | species<-c_preds {cc:.3f}")
    parts=list(Ss[0]["species_from_part_cpreds"].keys())
    PARTV=pd.DataFrame({"species_code":[np.mean([s["species_from_part_cpreds"][p]["acc"] for s in Ss]) for p in parts],
                        "n_variants":[Ss[0]["species_from_part_cpreds"][p]["n_variants"] for p in parts]}, index=parts)
    PARTV=PARTV.sort_values("species_code", ascending=False); display(PARTV.round(3))
    fig,ax=plt.subplots(1,2,figsize=(10,3.2))
    ax[0].bar(["z","c_preds"],[zc,cc],color=CBM_C); ax[0].axhline(ch,ls="--",color="k",label="chance")
    ax[0].set_ylim(0,1); ax[0].set_title("species recoverable from bottleneck"); ax[0].legend()
    ax[1].bar(PARTV.index, PARTV.species_code, color=CBM_C); ax[1].axhline(ch,ls="--",color="k")
    ax[1].set_title("species from EACH part's concepts"); plt.setp(ax[1].get_xticklabels(),rotation=30,ha="right")

## 3 · Line them up — does per-part retention track the mechanism?
Per part: visible-only `retained_frac` vs species-code vs n_variants (seed-averaged).
Correlation, not proof. Note wing: high species-code, low retention — species-coding is
*necessary* not *sufficient*; the part must also be the shortcut (small/hard-to-see).

In [ ]:
if G is not None and PARTV is not None:
    M = per_part_seedagg(G, visible_only=True)[["mean"]].rename(columns={"mean":"retained_frac"}).join(PARTV)
    display(M.round(3))
    fig,ax=plt.subplots(1,2,figsize=(10,3.4))
    for k,(x,xl) in enumerate({"species_code":"species from part's concepts","n_variants":"# concept variants"}.items()):
        ax[k].scatter(M[x], M.retained_frac, color=CBM_C)
        for p,r in M.iterrows(): ax[k].annotate(p,(r[x],r.retained_frac),fontsize=8)
        ax[k].set_xlabel(xl); ax[k].set_ylabel("retained_frac (visible-only)")
    plt.tight_layout(); print("tail: top on species-code AND retention = mechanism realized.")
else:
    print("[pending] needs sections 1 and 2.")

## Takeaway
CBM concepts near-perfect on-distribution, yet a **removed** part's concept is retained
(tail), concentrated where the bottleneck most encodes species AND the part is hardest to
see. Notebook 03 asks whether MCBM minimality removes this.

## How `retained_frac` is read as a backwash measurement
No axis is labelled "backwash"; the computed number is
**`retained_frac = P(concept | part removed) / P(concept | intact)`**, on **visible-only**
removals (the part actually left the render). Grounded → collapses to ~0; backwashed →
stays ~1. `retained_frac` is the metric; "concept–class backwash" is the interpretation.